# EP3: Optimización de Hiperparámetros y Técnicas de Ensamble
### MLY1101 Machine Learning - Evaluación Parcial N°3
**Caso B:** Predicción de precios de viviendas (Ames Housing Dataset)
**Equipo N°7:** Nicolas Carrasco, Martin Corvalan, Martin Naranjo

**Indicadores de esta evaluación:**
- IE9: Optimización de hiperparámetros del modelo (20%)
- IE10: Implementación de técnicas de ensamble (30%)
- IE11: Evaluación de la capacidad de generalización del modelo (20%)
- IE12: Selección y justificación de la solución analítica óptima (30%)

**Antecedentes:** Este notebook se construye sobre el trabajo de las EV1 y EV2. Los modelos base (Ridge, Random Forest, Gradient Boosting) ya fueron entrenados y evaluados en `Housing_EV1_EV2.ipynb`. Aquí se reutilizan **la misma división train/test (80/20, `random_state=42`), el mismo preprocesador y el mismo esquema de validación cruzada (3 pliegues)**, de modo que todos los resultados sean comparables con los del README.

## Plan de trabajo

1. **Paso 1:** Nombrar explícitamente las técnicas de ensamble existentes (Bagging y Boosting)
2. **Paso 2:** Implementar un ensamble que combine modelos distintos (VotingRegressor) y **medir su aporte** frente a los modelos individuales
3. **Paso 3:** Justificar los hiperparámetros y documentar con precisión qué se exploró
4. **Paso 4:** Evaluar la capacidad de generalización (brecha train/test + validación cruzada)
5. **Paso 5:** Actualizar la justificación de la solución óptima

---
# ══════════════════════════════════════════
# PASO 1: Técnicas de Ensamble - Bagging y Boosting
# ══════════════════════════════════════════

Un **ensamble** combina varios modelos base para lograr predicciones más estables y precisas que las de un modelo individual. En el proyecto ya se usaban dos familias de ensamble (Random Forest y Gradient Boosting); aquí se nombran explícitamente y, en el Paso 2, se **cuantifica su aporte** frente a un árbol de decisión individual.

### Random Forest = Ensamble por Bagging (Bootstrap Aggregating)

**Cómo funciona:**
- Entrena múltiples árboles de decisión **en paralelo e independientes** (200 árboles)
- Cada árbol usa una muestra aleatoria con reemplazo de los datos (bootstrap)
- En cada división solo considera un subconjunto aleatorio de variables (`max_features=0,33`)
- **Predicción final:** promedio de las predicciones de todos los árboles
- **Objetivo:** reducir la **varianza** del árbol individual promediando árboles distintos entre sí

**Resultado en nuestro proyecto (conjunto de prueba, README sección 12.1):**
- RMSE test: US$ 25.294 | R² test: 0,921
- R² entrenamiento: 0,976 → **brecha train-test: 0,055** (la mayor de los tres modelos)

### Gradient Boosting = Ensamble por Boosting

**Cómo funciona:**
- Construye árboles **de forma secuencial**: cada árbol nuevo ajusta los **residuos** del conjunto anterior
- Los árboles son poco profundos (`max_depth=4`) y cada corrección se pondera por `learning_rate=0,05`
- Regularización adicional: `subsample=0,8` (80% de las filas por árbol) y `max_features=0,5`
- **Predicción final:** suma ponderada de las predicciones de todos los árboles (500 árboles)
- **Objetivo:** reducir el **sesgo** mejorando progresivamente el modelo

**Resultado en nuestro proyecto (conjunto de prueba, README sección 12.1):**
- RMSE test: US$ 19.246 | R² test: 0,954
- R² entrenamiento: 0,988 → **brecha train-test: 0,034**

> **Nota sobre sobreajuste:** GB alcanza el mayor R² de entrenamiento (0,988), pero su brecha con el test (0,034) es **menor** que la de Random Forest (0,055). Es decir, con `learning_rate` bajo y `subsample < 1`, el boosting está bien regularizado; no es el modelo más sobreajustado de los tres.

### Comparación: Bagging vs Boosting

| Aspecto | Random Forest (Bagging) | Gradient Boosting (Boosting) |
|---------|-------------------------|-------------------------------|
| Construcción | Árboles en paralelo (independientes) | Árboles en secuencia (corrigen errores) |
| Predicción final | Promedio de árboles | Suma ponderada de correcciones |
| Objetivo principal | Reducir varianza | Reducir sesgo |
| Velocidad | Rápido (paralelizable) | Más lento (secuencial) |
| RMSE test (nuestro caso) | US$ 25.294 | US$ 19.246 (mejor) |
| R² test (nuestro caso) | 0,921 | 0,954 (mejor) |
| R² entrenamiento | 0,976 | 0,988 |
| Brecha R² train-test | 0,055 | 0,034 (menor) |

El **aporte de cada técnica frente a un modelo individual** (un solo árbol de decisión) se mide en la tabla del Paso 2.

---
# ══════════════════════════════════════════
# PASO 2: Ensamble Combinado (VotingRegressor) y medición del aporte
# ══════════════════════════════════════════

### ¿Por qué VotingRegressor?

**VotingRegressor** combina las predicciones de modelos de regresión **de naturaleza distinta**:
- **Ridge** (lineal regularizado) + **Random Forest** (bagging) + **Gradient Boosting** (boosting)
- Predicción final = promedio ponderado de las predicciones de los modelos base
- A diferencia de RF y GB (muchos árboles del mismo tipo), aquí se combinan **familias de modelos diferentes**, cuyos errores pueden no estar correlacionados
- Todo se entrena sobre `log1p(SalePrice)`; las métricas se reportan en dólares con `expm1`

**Alternativa:** `StackingRegressor` (un meta-modelo aprende a combinar), más complejo; queda como mejora futura.

**Qué se mide:** el aporte del ensamble se evalúa contra (a) un árbol de decisión individual (para bagging y boosting) y (b) Ridge, el mejor modelo individual (para el Voting).

In [ ]:
# Importar librerías necesarias
import itertools
import pandas as pd
import numpy as np
import joblib
from sklearn.base import clone
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, KFold, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

In [ ]:
# Cargar datos limpios
df = pd.read_csv('../data/housing_clean.csv')
print(f"Dataset cargado: {df.shape}")

In [ ]:
# Cargar el preprocesador de EV1 (se clonará en cada pipeline, igual que en EV2)
preprocessor = joblib.load('../models/preprocessor_pipeline.pkl')
print("Preprocesador cargado.")

In [ ]:
# Separar X e y (misma división que EV1/EV2: 80/20, random_state=42)
X = df.drop(columns=['SalePrice', 'SalePrice_log'])
y = df['SalePrice_log']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

In [ ]:
# Modelos con los hiperparámetros elegidos en EV2 (README sección 11.2)
ridge = Ridge(alpha=30, random_state=RANDOM_STATE)
rf = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=2,
                           max_features=0.33, random_state=RANDOM_STATE, n_jobs=-1)
gb = GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                               subsample=0.8, max_features=0.5, random_state=RANDOM_STATE)

# Pipelines: preprocesador clonado + modelo (sin fuga de información en la validación cruzada)
pipe_ridge = Pipeline([('prep', clone(preprocessor)), ('model', ridge)])
pipe_rf    = Pipeline([('prep', clone(preprocessor)), ('model', rf)])
pipe_gb    = Pipeline([('prep', clone(preprocessor)), ('model', gb)])

# Referencia sin ensamble: un único árbol de decisión
pipe_tree  = Pipeline([('prep', clone(preprocessor)),
                       ('model', DecisionTreeRegressor(random_state=RANDOM_STATE))])

for p in (pipe_ridge, pipe_rf, pipe_gb, pipe_tree):
    p.fit(X_train, y_train)
print("Modelos base y árbol de referencia entrenados.")

In [ ]:
# Crear y entrenar el VotingRegressor (pesos iguales como punto de partida)
voting_regressor = VotingRegressor(
    estimators=[
        ('ridge', clone(pipe_ridge)),
        ('random_forest', clone(pipe_rf)),
        ('gradient_boosting', clone(pipe_gb))
    ],
    weights=[1, 1, 1]
)
voting_regressor.fit(X_train, y_train)
print("VotingRegressor (pesos 1-1-1) entrenado.")

In [ ]:
# Función para calcular métricas en dólares (la escala en que se definieron los umbrales de EV1)
def metricas_en_dolares(y_log_real, y_log_pred):
    y_real = np.expm1(y_log_real)
    y_pred = np.expm1(y_log_pred)
    return {
        'R2': r2_score(y_real, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_real, y_pred)),
        'MAE': mean_absolute_error(y_real, y_pred),
        'MAPE': mean_absolute_percentage_error(y_real, y_pred) * 100
    }

# Mismo esquema de validación cruzada que EV2: 3 pliegues, semilla 42
cv_kf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

### Tabla comparativa: modelo individual vs ensambles

Incluye validación cruzada sobre el conjunto de entrenamiento (RMSE en escala log, el mismo criterio con que se eligieron los hiperparámetros en EV2) y desempeño en el conjunto de prueba (en dólares).
**Verificación de consistencia:** el RMSE CV de Ridge debería reproducir ≈ 0,1229; el de Random Forest ≈ 0,1375 y el de Gradient Boosting ≈ 0,1244 (README sección 11.2).

In [ ]:
modelos = {
    'Árbol individual (referencia)': pipe_tree,
    'Ridge (modelo individual)':     pipe_ridge,
    'Random Forest (bagging)':       pipe_rf,
    'Gradient Boosting (boosting)':  pipe_gb,
    'Voting: Ridge+RF+GB (1-1-1)':   voting_regressor,
}

filas = []
for nombre, m in modelos.items():
    cv = -cross_val_score(clone(m), X_train, y_train, cv=cv_kf,
                          scoring='neg_root_mean_squared_error')
    mt  = metricas_en_dolares(y_test,  m.predict(X_test))
    mtr = metricas_en_dolares(y_train, m.predict(X_train))
    filas.append({
        'Modelo': nombre,
        'RMSE CV log (media)': cv.mean(),
        'RMSE CV log (desv.)': cv.std(),
        'R² train': mtr['R2'],
        'R² test': mt['R2'],
        'Brecha R²': mtr['R2'] - mt['R2'],
        'RMSE test (US$)': mt['RMSE'],
        'MAE test (US$)': mt['MAE'],
        'MAPE test (%)': mt['MAPE'],
    })

tabla_final = pd.DataFrame(filas).set_index('Modelo').round(4)
display(tabla_final)

### Selección de pesos del Voting (sin usar el conjunto de prueba)

Se calculan predicciones **fuera de pliegue (OOF)** de cada modelo base solo con el conjunto de entrenamiento, y se prueban combinaciones de pesos con ellas. Así los pesos se eligen sin que el conjunto de prueba participe.

In [ ]:
oof = {n: cross_val_predict(clone(m), X_train, y_train, cv=cv_kf)
       for n, m in [('ridge', pipe_ridge), ('rf', pipe_rf), ('gb', pipe_gb)]}

res = {}
for w in itertools.product([0, 1, 2, 3], repeat=3):
    if sum(w) == 0:
        continue
    norm = tuple(round(x / sum(w), 4) for x in w)      # evita duplicados como (1,1,1) y (2,2,2)
    if norm in res:
        continue
    p = (w[0]*oof['ridge'] + w[1]*oof['rf'] + w[2]*oof['gb']) / sum(w)
    res[norm] = (w, np.sqrt(mean_squared_error(y_train, p)))

top = sorted(res.values(), key=lambda t: t[1])[:6]
print("Mejores pesos (Ridge, RF, GB) por RMSE OOF en escala log, solo con train:")
for w, r in top:
    print(f"  {w}  ->  {r:.4f}")
print(f"Pesos iguales (1,1,1) -> {res[(0.3333, 0.3333, 0.3333)][1]:.4f}")

In [ ]:
# Voting con los pesos óptimos encontrados en train (el test se usa UNA sola vez, para reportar)
pesos_opt = list(top[0][0])
print(f"Pesos óptimos (Ridge, RF, GB): {pesos_opt}")

voting_opt = VotingRegressor(
    estimators=[
        ('ridge', clone(pipe_ridge)),
        ('random_forest', clone(pipe_rf)),
        ('gradient_boosting', clone(pipe_gb))
    ],
    weights=pesos_opt
)
voting_opt.fit(X_train, y_train)

mt  = metricas_en_dolares(y_test,  voting_opt.predict(X_test))
mtr = metricas_en_dolares(y_train, voting_opt.predict(X_train))
rmse_cv_opt = np.sqrt(mean_squared_error(y_train, sum(w*oof[k] for w, k in zip(pesos_opt, ['ridge', 'rf', 'gb'])) / sum(pesos_opt)))

fila_opt = pd.DataFrame([{
    'RMSE CV log (media)': rmse_cv_opt,
    'RMSE CV log (desv.)': np.nan,
    'R² train': mtr['R2'], 'R² test': mt['R2'], 'Brecha R²': mtr['R2'] - mt['R2'],
    'RMSE test (US$)': mt['RMSE'], 'MAE test (US$)': mt['MAE'], 'MAPE test (%)': mt['MAPE'],
}], index=[f'Voting optimizado {tuple(pesos_opt)}'])

tabla_final = pd.concat([tabla_final, fila_opt.round(4)])
tabla_final.index.name = 'Modelo'
display(tabla_final)

### Análisis del aporte de los ensambles (IE10)

> **[COMPLETAR con los resultados de la tabla anterior]**
>
> - **Bagging:** Random Forest frente al árbol individual: ¿cuánto mejora el RMSE CV y el R² test? ¿Baja la desviación entre pliegues? (reducción de varianza)
> - **Boosting:** Gradient Boosting frente al árbol individual: ¿cuánto mejora? (corrección secuencial de residuos)
> - **Voting:** frente a Ridge (mejor modelo individual) y frente a los ensambles de un solo tipo: ¿mejora, empata o empeora el RMSE test? ¿Cambia la brecha train-test?
> - **Interpretación:** si el Voting no supera a Ridge, explicar por qué (Ridge ya captura gran parte de la relación tras `log1p`, y RF, el modelo más débil, aporta poca información adicional). Si lo supera, cuantificar la mejora en dólares.

---
# ══════════════════════════════════════════
# PASO 3: Optimización y justificación de hiperparámetros
# ══════════════════════════════════════════

**Esquema común (EV2):** los hiperparámetros se eligieron con **validación cruzada de 3 pliegues sobre el conjunto de entrenamiento**, minimizando el RMSE sobre `log1p(SalePrice)`. El conjunto de prueba no participó en la elección. Cada modelo es un `Pipeline` (preprocesador + modelo), de modo que imputación, escalado y One-Hot se reajustan dentro de cada pliegue.

### Resumen de lo explorado

| Modelo | Estrategia | Espacio de búsqueda | Combinaciones evaluadas | Elegido | RMSE CV (log) |
|---|---|---|---|---|---|
| Ridge | `GridSearchCV` | `alpha` ∈ {0,1; 0,3; 1; 3; 10; 30; 100} | 7 de 7 | `alpha`=30 | 0,1229 |
| Random Forest | `RandomizedSearchCV` (`n_iter=5`) | `max_depth` ∈ {None, 15, 25} × `min_samples_leaf` ∈ {1, 2, 4} × `max_features` ∈ {0,2; 0,33; 0,5} | **5 de 27** (≈ 19%) | `max_depth`=15, `min_samples_leaf`=2, `max_features`=0,33 | 0,1375 |
| Gradient Boosting | `GridSearchCV` | `n_estimators` ∈ {300, 500} × `max_depth` ∈ {3, 4} | 4 de 4 | `n_estimators`=500, `max_depth`=4 | 0,1244 |
| Voting | Búsqueda de pesos sobre predicciones OOF de train (Paso 2) | pesos enteros 0-3 por modelo | todas las combinaciones | ver Paso 2 | ver Paso 2 |

> **Transparencia sobre el alcance:** en Gradient Boosting, `learning_rate=0,05`, `subsample=0,8` y `max_features=0,5` se **fijaron** a valores conservadores (no se optimizaron); en Random Forest, `n_estimators=200` también es fijo. En Random Forest la búsqueda fue aleatoria y cubrió 5 de las 27 combinaciones posibles.

### Ridge: `alpha`
- `alpha` controla la fuerza de la regularización L2.
- Con 314 columnas (tras One-Hot) para 2.341 filas y variables muy correlacionadas (EV1, multicolinealidad), una regularización casi nula arriesga coeficientes inestables; una muy alta subajusta.
- `alpha=30` está en el interior del rango explorado (no en un borde), lo que indica que el óptimo fue encontrado y no truncado por la grilla.

### Random Forest
- `max_depth=15`: limita la profundidad para controlar el sobreajuste y deja suficiente complejidad.
- `min_samples_leaf=2`: exige al menos 2 muestras por hoja, evita divisiones demasiado específicas.
- `max_features=0,33`: cada división ve 1/3 de las variables, lo que decorrelaciona los árboles (esencial en bagging).
- `n_estimators=200` (fijo): suficiente para estabilizar el promedio sin costo computacional excesivo.

### Gradient Boosting
- `n_estimators=500` con `learning_rate=0,05` (fijo): tasa conservadora, cada árbol corrige solo una fracción pequeña del error, y se compensa con más árboles.
- `max_depth=4`: árboles poco profundos ("weak learners"), que reducen el sobreajuste propio del boosting.
- `subsample=0,8` y `max_features=0,5` (fijos): aleatorización que actúa como regularización (stochastic gradient boosting).

### VotingRegressor: pesos
- Los modelos base ya estaban optimizados individualmente; el ensamble solo requiere decidir cuánto confiar en cada uno.
- Los pesos se eligieron con predicciones fuera de pliegue sobre el conjunto de entrenamiento (Paso 2), sin tocar el test. Random Forest es el modelo base más débil en validación cruzada (0,1375 frente a 0,1229 de Ridge y 0,1244 de GB), por lo que no corresponde darle más peso por defecto.

### (Opcional) Búsqueda ampliada para Gradient Boosting

La grilla original de GB solo varió `n_estimators` y `max_depth`. Esta celda amplía la búsqueda a `learning_rate`, `subsample`, `max_features` y `min_samples_leaf` con `RandomizedSearchCV`, para **demostrar si la optimización mejora el rendimiento** (IE9). Tarda varios minutos (GB de hasta 800 árboles × 12 combinaciones × 3 pliegues), por eso está desactivada por defecto: cambiar `EJECUTAR_BUSQUEDA_GB = True` para correrla.

In [ ]:
EJECUTAR_BUSQUEDA_GB = False   # cambiar a True para correr la búsqueda ampliada

if EJECUTAR_BUSQUEDA_GB:
    pipe_gb_busq = Pipeline([('prep', clone(preprocessor)),
                             ('model', GradientBoostingRegressor(random_state=RANDOM_STATE))])
    espacio_gb = {
        'model__n_estimators':     [300, 500, 800],
        'model__learning_rate':    [0.02, 0.03, 0.05, 0.08],
        'model__max_depth':        [3, 4, 5],
        'model__subsample':        [0.7, 0.8, 0.9],
        'model__max_features':     [0.3, 0.5, 0.7],
        'model__min_samples_leaf': [1, 3, 5],
    }
    busq_gb = RandomizedSearchCV(pipe_gb_busq, espacio_gb, n_iter=12, cv=cv_kf,
                                 scoring='neg_root_mean_squared_error',
                                 random_state=RANDOM_STATE, n_jobs=-1)
    busq_gb.fit(X_train, y_train)

    print("Mejores hiperparámetros (búsqueda ampliada):", busq_gb.best_params_)
    print(f"RMSE CV log: {-busq_gb.best_score_:.4f}   (GB original de EV2: 0.1244)")
    mt = metricas_en_dolares(y_test, busq_gb.best_estimator_.predict(X_test))
    print({k: round(v, 4) for k, v in mt.items()})
    print("GB original en test -> RMSE 19.246 | R² 0.954")
else:
    print("Búsqueda ampliada desactivada (EJECUTAR_BUSQUEDA_GB = False).")